# 美国工资税费估算（2024 / 2025）

估算 2024 / 2025 年华盛顿州（WA）和加州（CA）的工资税费、401(k) / HSA 供款及扣除后的年度和月均余额。金额以美元表示。


In [1]:
INF = float('inf')

TAX_DATA = {
    2024: {
        'federal': {
            'single':    {'std_deduction': 14_600, 'brackets': [
                (11_600, 0.10), (47_150, 0.12), (100_525, 0.22), (191_950, 0.24),
                (243_725, 0.32), (609_350, 0.35), (INF, 0.37)]},
            'married':   {'std_deduction': 29_200, 'brackets': [
                (23_200, 0.10), (94_300, 0.12), (201_050, 0.22), (383_900, 0.24),
                (487_450, 0.32), (731_200, 0.35), (INF, 0.37)]},
            'household': {'std_deduction': 21_900, 'brackets': [
                (16_550, 0.10), (63_100, 0.12), (100_500, 0.22), (191_950, 0.24),
                (243_700, 0.32), (609_350, 0.35), (INF, 0.37)]},
        },
        'ss_wage_base': 168_600, '401k_limit': 23_000,
        'hsa_limit': {'single': 4_150, 'family': 8_300},
        'states': {
            'WA': {},
            'CA': {
                'single': {'std_deduction': 5_540, 'brackets': [
                    (10_756, 0.010), (25_499, 0.020), (40_245, 0.040), (55_866, 0.060),
                    (70_606, 0.080), (360_659, 0.093), (432_787, 0.103), (721_314, 0.113), (INF, 0.123)]},
                'married': {'std_deduction': 11_080, 'brackets': [
                    (21_512, 0.010), (50_998, 0.020), (80_490, 0.040), (111_732, 0.060),
                    (141_212, 0.080), (721_318, 0.093), (865_574, 0.103), (1_442_628, 0.113), (INF, 0.123)]},
                'household': {'std_deduction': 11_080, 'brackets': [
                    (21_527, 0.010), (51_000, 0.020), (65_744, 0.040), (81_364, 0.060),
                    (96_107, 0.080), (490_493, 0.093), (588_593, 0.103), (980_987, 0.113), (INF, 0.123)]},
                'mental_health_threshold': 1_000_000, 'mental_health_rate': 0.01, 'sdi_rate': 0.011,
            },
        },
    },
    2025: {
        'federal': {
            'single':    {'std_deduction': 15_750, 'brackets': [
                (11_925, 0.10), (48_475, 0.12), (103_350, 0.22), (197_300, 0.24),
                (250_525, 0.32), (626_350, 0.35), (INF, 0.37)]},
            'married':   {'std_deduction': 31_500, 'brackets': [
                (23_850, 0.10), (96_950, 0.12), (206_700, 0.22), (394_600, 0.24),
                (501_050, 0.32), (751_600, 0.35), (INF, 0.37)]},
            'household': {'std_deduction': 23_625, 'brackets': [
                (17_000, 0.10), (64_850, 0.12), (103_350, 0.22), (197_300, 0.24),
                (250_500, 0.32), (626_350, 0.35), (INF, 0.37)]},
        },
        'ss_wage_base': 176_100, '401k_limit': 23_500,
        'hsa_limit': {'single': 4_300, 'family': 8_550},
        'states': {
            'WA': {},
            'CA': {
                'single': {'std_deduction': 5_706, 'brackets': [
                    (11_079, 0.010), (26_264, 0.020), (41_452, 0.040), (57_542, 0.060),
                    (72_724, 0.080), (371_479, 0.093), (445_771, 0.103), (742_953, 0.113), (INF, 0.123)]},
                'married': {'std_deduction': 11_412, 'brackets': [
                    (22_158, 0.010), (52_528, 0.020), (82_904, 0.040), (115_084, 0.060),
                    (145_448, 0.080), (742_958, 0.093), (891_542, 0.103), (1_485_906, 0.113), (INF, 0.123)]},
                'household': {'std_deduction': 11_412, 'brackets': [
                    (22_173, 0.010), (52_530, 0.020), (67_716, 0.040), (83_805, 0.060),
                    (98_990, 0.080), (505_208, 0.093), (606_251, 0.103), (1_010_417, 0.113), (INF, 0.123)]},
                'mental_health_threshold': 1_000_000, 'mental_health_rate': 0.01, 'sdi_rate': 0.012,
            },
        },
    },
}

SS_RATE = 0.062
MEDICARE_RATE = 0.0145
ADDL_MEDICARE_RATE = 0.009
ADDL_MEDICARE_THRESHOLD = {'single': 200_000, 'married': 250_000, 'household': 200_000}

In [2]:
def progressive_tax(taxable, brackets):
    tax, prev = 0.0, 0
    for upper, rate in brackets:
        if taxable <= prev:
            break
        tax += (min(taxable, upper) - prev) * rate
        prev = upper
    return tax

def take_home_pay(yearly_gross, state, year, status='single',
                  hsa_contribution=0, contribution_401k=0):
    data = TAX_DATA[year]
    fed = data['federal'][status]
    st = data['states'][state.upper()]

    contrib_401k = min(contribution_401k, data['401k_limit'])
    hsa_limit = data['hsa_limit']['family' if status == 'married' else 'single']
    contrib_hsa = min(hsa_contribution, hsa_limit)

    federal_taxable = max(0, yearly_gross - contrib_401k - contrib_hsa - fed['std_deduction'])
    federal_tax = progressive_tax(federal_taxable, fed['brackets'])

    state_tax = sdi = 0.0
    if st:
        st_s = st.get(status, st.get('single', {}))
        state_taxable = max(0, yearly_gross - contrib_401k - st_s.get('std_deduction', 0))
        state_tax = progressive_tax(state_taxable, st_s['brackets'])
        mh_thresh = st.get('mental_health_threshold', INF)
        if state_taxable > mh_thresh:
            state_tax += (state_taxable - mh_thresh) * st['mental_health_rate']
        sdi = yearly_gross * st.get('sdi_rate', 0)

    fica_wages = yearly_gross - contrib_hsa
    ss_tax = min(fica_wages, data['ss_wage_base']) * SS_RATE
    medicare_tax = fica_wages * MEDICARE_RATE
    addl_medicare = max(0, fica_wages - ADDL_MEDICARE_THRESHOLD[status]) * ADDL_MEDICARE_RATE

    total_tax = federal_tax + state_tax + sdi + ss_tax + medicare_tax + addl_medicare
    total_pretax = contrib_401k + contrib_hsa
    net = yearly_gross - total_tax - total_pretax

    return dict(gross=yearly_gross, contrib_401k=contrib_401k, contrib_hsa=contrib_hsa,
                federal_tax=federal_tax, state_tax=state_tax, sdi=sdi,
                ss_tax=ss_tax, medicare_tax=medicare_tax, addl_medicare=addl_medicare,
                total_tax=total_tax, total_pretax=total_pretax, net=net)

# 合报辅助函数：HSA 限额各分一半，有工资者计供款；所得税按工资比例分摊。
def take_home_mfj(gross1, gross2, state, year,
                  hsa_contribution=0, contribution_401k=0):
    data = TAX_DATA[year]
    fed = data['federal']['married']
    st = data['states'][state.upper()]

    limit_401k = data['401k_limit']
    limit_hsa_half = data['hsa_limit']['family'] / 2

    def person_pretax(g):
        if g <= 0:
            return 0, 0
        return min(contribution_401k, limit_401k), min(hsa_contribution, limit_hsa_half)

    k1, h1 = person_pretax(gross1)
    k2, h2 = person_pretax(gross2)

    combined = gross1 + gross2
    fed_taxable = max(0, combined - (k1+k2) - (h1+h2) - fed['std_deduction'])
    fed_tax = progressive_tax(fed_taxable, fed['brackets'])

    state_tax = sdi = 0.0
    if st:
        st_m = st.get('married', st.get('single', {}))
        state_taxable = max(0, combined - (k1+k2) - st_m.get('std_deduction', 0))
        state_tax = progressive_tax(state_taxable, st_m['brackets'])
        mh = st.get('mental_health_threshold', INF)
        if state_taxable > mh:
            state_tax += (state_taxable - mh) * st['mental_health_rate']
        sdi = combined * st.get('sdi_rate', 0)

    def person_fica(g, hsa):
        w = max(0, g - hsa)
        return min(w, data['ss_wage_base']) * SS_RATE, w * MEDICARE_RATE

    ss1, med1 = person_fica(gross1, h1)
    ss2, med2 = person_fica(gross2, h2)
    combined_fica = max(0, gross1 - h1) + max(0, gross2 - h2)
    addl_med = max(0, combined_fica - ADDL_MEDICARE_THRESHOLD['married']) * ADDL_MEDICARE_RATE

    ratio = gross1 / combined if combined else 0.5
    def split(g, r, ss, med, k, h):
        pt = k + h
        ft = fed_tax * r; stt = state_tax * r; sd = sdi * r; am = addl_med * r
        tt = ft + stt + sd + ss + med + am
        return dict(gross=g, contrib_401k=k, contrib_hsa=h,
                    federal_tax=ft, state_tax=stt, sdi=sd,
                    ss_tax=ss, medicare_tax=med, addl_medicare=am,
                    total_tax=tt, total_pretax=pt, net=g - tt - pt)

    return split(gross1, ratio, ss1, med1, k1, h1), split(gross2, 1 - ratio, ss2, med2, k2, h2)

def show(r, state, year, status='single'):
    g = r['gross']
    def row(label, val, rate=True):
        pct = f"  ({val/g*100:5.1f}%)" if rate else ""
        print(f"  {label:<28s} ${val:>12,.2f}{pct}")
    print(f"{'\u2500'*62}")
    print(f"  {year} \u00b7 {state} \u00b7 {status.title()}")
    print(f"{'\u2500'*62}")
    print('  Estimate of listed items only; not a paystub.')
    row('Gross Income', g, rate=False)
    print()
    if r['contrib_401k']: row('401k Contribution',    r['contrib_401k'])
    if r['contrib_hsa']:  row('HSA Contribution',     r['contrib_hsa'])
    print()
    row('Federal Income Tax',   r['federal_tax'])
    if r['state_tax']: row('State Income Tax',    r['state_tax'])
    if r['sdi']:       row('CA SDI',              r['sdi'])
    row('Social Security',      r['ss_tax'])
    row('Medicare',             r['medicare_tax'])
    if r['addl_medicare']: row('Additional Medicare', r['addl_medicare'])
    print(f"{'\u2500'*62}")
    row('Included Taxes',            r['total_tax'])
    row('401k + HSA Contributions', r['total_pretax'])
    print(f"{'\u2500'*62}")
    row('Est. Annual Remaining',     r['net'])
    row('Est. Monthly Remaining',    r['net'] / 12, rate=False)
    print(f"  {'Included Taxes / Gross':<28s} {r['total_tax']/g*100:>12.1f}%")
    print(f"{'\u2500'*62}")

In [3]:
# 修改参数后按顺序运行各代码单元；工资与员工供款均按年填写。
yearly_gross       = 120_000 # 年工资，美元
state              = "WA"    # WA = 华盛顿州；CA = 加州
year               = 2025    # 收入所属税年：2024 或 2025
status             = "single" # single = 单身；married = 夫妻合报；household = 户主

# INF 表示取模型供款上限；HSA 假设全年合资格且经工资预扣。
# HSA 简化为 married 用家庭限额、其余用个人限额；填数值时应扣除雇主已供款。
hsa_contribution   = INF
contribution_401k  = INF     # 员工传统 401(k) 供款

# 采用标准扣除和展示税项；WA PFML、WA Cares、医保等扣款可另从余额扣除。
# 月余额 = 年余额 / 12。
r = take_home_pay(yearly_gross, state, year, status, hsa_contribution, contribution_401k)
show(r, state, year, status)

──────────────────────────────────────────────────────────────
  2025 · WA · Single
──────────────────────────────────────────────────────────────
  Estimate of listed items only; not a paystub.
  Gross Income                 $  120,000.00

  401k Contribution            $   23,500.00  ( 19.6%)
  HSA Contribution             $    4,300.00  (  3.6%)

  Federal Income Tax           $   11,733.00  (  9.8%)
  Social Security              $    7,173.40  (  6.0%)
  Medicare                     $    1,677.65  (  1.4%)
──────────────────────────────────────────────────────────────
  Included Taxes               $   20,584.05  ( 17.2%)
  401k + HSA Contributions     $   27,800.00  ( 23.2%)
──────────────────────────────────────────────────────────────
  Est. Annual Remaining        $   71,615.95  ( 59.7%)
  Est. Monthly Remaining       $    5,968.00
  Included Taxes / Gross               17.2%
──────────────────────────────────────────────────────────────


税表核对：2026-10-04。来源：[IRS 标准扣除额](https://www.irs.gov/publications/p501) · [FTB 2024](https://www.ftb.ca.gov/forms/2024/2024-540-booklet.html) · [FTB 2025](https://www.ftb.ca.gov/forms/2025/2025-540-booklet.html) · [IRS HSA 规则](https://www.irs.gov/publications/p969)。
